# Sprint 8 — GenAI Insight Layer

**Design principle:** The analytics layer calculates and decides; the GenAI layer explains.

This notebook demonstrates the batch-generated GenAI insight layer. It calls reusable pipeline functions from `src/genai/` and displays their outputs. It does not duplicate pipeline logic or make unnecessary live API calls.

All insights shown here are generated by the deterministic rule-based fallback because no API key is configured in the demonstration environment. The validation and grounding logic is identical regardless of generation path.

In [ ]:
from pathlib import Path
import os
import sys
import json

import pandas as pd

# Resolve project root regardless of where the notebook is launched from
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'API key configured: {bool(os.environ.get("OPENAI_API_KEY"))}')

## 1. Architecture

The GenAI layer sits after the analytics layer in the pipeline. It reads a single canonical mart and writes structured JSON outputs. It does not compute any metrics and does not choose any actions — those are determined by the analytics pipeline.

**Report-level flow:**

```
mart_report_analytics.csv
        ↓
Privacy-safe report context (34-field allowlist)
        ↓
Versioned report prompt  (report_insight_v1)
        ↓
Structured report insight  (7 required narrative fields)
        ↓
Validation and fallback  (schema + grounding + direction + safety)
        ↓
report_ai_insights.json
```

**Portfolio-level flow:**

```
mart_report_analytics.csv
        ↓
Deterministic portfolio aggregates (usage / forecast / model health / engagement)
        ↓
Versioned portfolio prompt  (portfolio_insight_v1)
        ↓
Structured portfolio insight  (8 required narrative fields)
        ↓
Validation and fallback
        ↓
portfolio_ai_insight.json
```

## 2. Canonical Input

In [ ]:
mart_path = PROJECT_ROOT / 'outputs' / 'analytics' / 'mart_report_analytics.csv'
mart = pd.read_csv(mart_path)
print(f'Mart: {len(mart)} reports x {len(mart.columns)} columns')
print(f'Analytics run ID:   {mart["analytics_run_id"].iloc[0]}')
print(f'Analytics as-of:    {mart["analytics_as_of_date"].iloc[0]}')

## 3. Report-Level Context

Only 34 fields from the 305-column mart are passed to the LLM — the `INSIGHT_CONTEXT_ALLOWLIST`. Privacy-suppressed engagement values are passed as `null` so the LLM knows to disclose the limitation rather than invent values.

In [ ]:
from src.genai.insight_generator import INSIGHT_CONTEXT_ALLOWLIST, build_mart_context

print(f'Allowlist field count:    {len(INSIGHT_CONTEXT_ALLOWLIST)}')
print(f'Excluded mart columns:    {len(mart.columns) - len(INSIGHT_CONTEXT_ALLOWLIST)}')
print(f'Allowlist fields: {sorted(INSIGHT_CONTEXT_ALLOWLIST)}')

In [ ]:
contexts = build_mart_context(mart)
print(f'Contexts built: {len(contexts)}')

# Representative context
sample_ctx = contexts[0]
print(f'\n--- Report context: {sample_ctx["report_id"]} ({sample_ctx["report_name"]}) ---')
for k, v in sample_ctx.items():
    print(f'  {k}: {v}')

In [ ]:
# Privacy-suppressed context example
suppressed = [c for c in contexts if c.get('privacy_suppression_status') == 'suppressed']
if suppressed:
    s = suppressed[0]
    print(f'Privacy-suppressed context: {s["report_id"]} ({s["report_name"]})')
    engagement_fields = [
        'unique_users_28d', 'active_user_direction_28d', 'returning_user_share_28d',
        'retained_user_rate_28d', 'lapse_rate_28d', 'overall_engagement_status'
    ]
    for f in engagement_fields:
        print(f'  {f}: {s.get(f)}  ← null because suppressed')
    print(f'  privacy_suppression_status:  {s.get("privacy_suppression_status")}')
    print(f'  privacy_suppressed_fields:   {s.get("privacy_suppressed_fields")}')
    print('\nThe LLM must disclose the suppression — it may not infer or estimate these values.')
else:
    print('No privacy-suppressed reports in this mart.')

## 4. Report-Level Pipeline

The pipeline checks for an existing stored output with a matching input hash before calling the API. Unchanged outputs are reused with no API call. If no API key is set, the deterministic fallback is used.

In [ ]:
from src.genai.insight_generator import generate_report_insights, save_insights
from src.genai.prompts import REPORT_INSIGHT_PROMPT_VERSION
from src.genai.insight_generator import DEFAULT_MODEL

print(f'Prompt version:        {REPORT_INSIGHT_PROMPT_VERSION}')
print(f'Default model:         {DEFAULT_MODEL}')
print(f'API key configured:    {bool(os.environ.get("OPENAI_API_KEY"))}')

insights = generate_report_insights(project_root=PROJECT_ROOT)
output_paths = save_insights(insights, project_root=PROJECT_ROOT)

print(f'\nGenerated {len(insights)} insights')
statuses = {}
for ins in insights:
    s = ins.get('generation_status', '?')
    statuses[s] = statuses.get(s, 0) + 1
print('Generation status distribution:', statuses)

## 5. Report-Level Output

One structured insight per report — 7 required narrative fields plus lineage.

In [ ]:
sample = insights[0]
narrative_fields = [
    'executive_summary', 'usage_insight', 'engagement_insight',
    'forecast_insight', 'model_confidence_note', 'recommended_action', 'evidence_limitations'
]
lineage_fields = [
    'report_id', 'report_name', 'analytics_run_id', 'analytics_as_of_date',
    'prompt_version', 'model_name', 'generation_status', 'validation_status',
    'input_hash', 'generated_at', 'genai_run_id'
]

print(f'=== {sample["report_id"]} — {sample["report_name"]} ===')
print('\n--- Narrative fields ---')
for f in narrative_fields:
    v = sample.get(f)
    if isinstance(v, list):
        print(f'[{f}]')
        for item in v:
            print(f'  • {item}')
    else:
        print(f'[{f}] {v}')

print('\n--- Lineage ---')
for f in lineage_fields:
    print(f'  {f}: {sample.get(f)}')

In [ ]:
# Summary table
pd.DataFrame([
    {
        'report_id':          ins['report_id'],
        'report_name':        ins['report_name'],
        'generation_status':  ins.get('generation_status'),
        'validation_status':  ins.get('validation_status'),
        'recommended_action': ins.get('recommended_action'),
    }
    for ins in insights
])

## 6. Portfolio Context

Portfolio aggregates are computed entirely from the analytics mart before any LLM call. No user-level data is included. The attention shortlist selects up to 5 non-monitoring reports, ranked deterministically by priority → status → report_id.

In [ ]:
from src.genai.portfolio_insights import build_portfolio_context, PORTFOLIO_INSIGHT_SHORTLIST_MAX
from src.genai.prompts import PORTFOLIO_INSIGHT_PROMPT_VERSION

portfolio_ctx = build_portfolio_context(mart)

print(f'Portfolio prompt version:  {PORTFOLIO_INSIGHT_PROMPT_VERSION}')
print(f'Total reports:             {portfolio_ctx["total_report_count"]}')
print(f'Attention shortlist cap:   {PORTFOLIO_INSIGHT_SHORTLIST_MAX}')

h = portfolio_ctx['historical_usage']
print('\nHistorical usage distribution:')
print(f'  growing:   {h["growing"]:3d}  ({h["growing_share_pct"]:.1f}%)')
print(f'  stable:    {h["stable"]:3d}  ({h["stable_share_pct"]:.1f}%)')
print(f'  declining: {h["declining"]:3d}  ({h["declining_share_pct"]:.1f}%)')
print(f'  inactive:  {h["inactive"]:3d}  ({h["inactive_share_pct"]:.1f}%)')

f = portfolio_ctx['forecast_outlook']
print('\nForecast outlook distribution:')
print(f'  growth_expected:              {f["growth_expected"]:3d}  ({f["growth_expected_share_pct"]:.1f}%)')
print(f'  decline_expected:             {f["decline_expected"]:3d}  ({f["decline_expected_share_pct"]:.1f}%)')
print(f'  high_or_very_high_uncertainty:{f["high_or_very_high_uncertainty"]:3d}  ({f["high_uncertainty_share_pct"]:.1f}%)')

mh = portfolio_ctx['model_health']
print('\nModel health status counts:', mh['status_counts'])

ds = portfolio_ctx['decision_support']
print('\nReview priority counts:        ', ds['review_priority_counts'])
print('Recommended action counts:     ', ds['recommended_action_counts'])

In [ ]:
# Attention shortlist — deterministic, capped at 5
shortlist = portfolio_ctx.get('attention_shortlist', [])
print(f'Attention shortlist: {len(shortlist)} item(s) (maximum {PORTFOLIO_INSIGHT_SHORTLIST_MAX})')
if shortlist:
    pd.DataFrame(shortlist)[[
        'report_id', 'report_name', 'overall_review_priority',
        'overall_report_status', 'recommended_report_action'
    ]]
else:
    print('All reports have recommended_report_action = continue_monitoring — no shortlist items.')

## 7. Portfolio Pipeline and Output

In [ ]:
from src.genai.portfolio_insights import run_portfolio_pipeline

portfolio_paths = run_portfolio_pipeline(project_root=PROJECT_ROOT)
portfolio_insight = json.loads(portfolio_paths['json'].read_text())

print(f'Generation status: {portfolio_insight["generation_status"]}')
print(f'Validation status: {portfolio_insight["validation_status"]}')
print(f'Report count:      {portfolio_insight["report_count"]}')
print(f'Analytics run ID:  {portfolio_insight["analytics_run_id"]}')

In [ ]:
portfolio_narrative = [
    'executive_summary', 'portfolio_usage_summary', 'portfolio_engagement_summary',
    'portfolio_forecast_summary', 'portfolio_model_health_summary',
    'priority_actions', 'positive_signals', 'evidence_limitations'
]

print('=== Portfolio Insight ===')
for f in portfolio_narrative:
    v = portfolio_insight.get(f)
    if isinstance(v, list):
        print(f'\n[{f}]')
        for item in v:
            print(f'  • {item}')
    else:
        print(f'\n[{f}] {v}')

print()
print('Note: all 30 reports in this synthetic mart have model_diagnostic_status = insufficient_evidence.')
print('The model_health_summary acknowledges this rather than inferring health. This is correct behaviour.')

## 8. Validation Examples

The same validation logic applies regardless of whether the insight was produced by the LLM or the rule-based fallback. The cells below call validation functions directly on synthetic dicts — **no live API calls**.

In [ ]:
from src.genai.evaluation import evaluate_report_insight

# Shared context for all examples
demo_ctx = contexts[0]  # growing, model evidence insufficient, not suppressed

# 1. Valid structured response
valid_insight = {
    'executive_summary':    'This report shows growing usage and healthy engagement.',
    'usage_insight':        'Usage has grown from 817 to 1007 views, a 23% increase over the prior 28-day window.',
    'engagement_insight':   '133 active users with healthy broad adoption.',
    'forecast_insight':     'A stable outlook is expected. Forecast uncertainty is high.',
    'model_confidence_note':'Model diagnostic evidence is insufficient.',
    'recommended_action':   'Continue monitoring.',
    'evidence_limitations': ['Model diagnostic evidence is insufficient — confidence intervals are unavailable.']
}
r = evaluate_report_insight(valid_insight, demo_ctx, case_id='demo_valid')
print(f'1. Valid response     → overall_pass: {r.overall_pass}  failures: {r.failure_reasons}')

In [ ]:
# 2. Unsupported numerical claim
bad_num = dict(valid_insight, engagement_insight='95% of users are returning users — exceptional loyalty.')
r2 = evaluate_report_insight(bad_num, demo_ctx, case_id='demo_bad_number')
print(f'2. Unsupported number → overall_pass: {r2.overall_pass}')
print(f'   numerical_pass: {r2.numerical_pass}')
print(f'   failure_reasons: {[x for x in r2.failure_reasons if "ungrounded" in x]}')

In [ ]:
# 3. Directional contradiction (growing context, declining language)
bad_dir = dict(valid_insight, usage_insight='Usage is declining significantly and falling each week.')
r3 = evaluate_report_insight(bad_dir, demo_ctx, case_id='demo_bad_direction')
print(f'3. Direction conflict  → overall_pass: {r3.overall_pass}')
print(f'   direction_pass: {r3.direction_pass}')
print(f'   failure_reasons: {[x for x in r3.failure_reasons if "direction" in x]}')

In [ ]:
# 4. Prohibited action phrase
bad_action = dict(valid_insight, recommended_action='We should retire this report — usage is too low.')
r4 = evaluate_report_insight(bad_action, demo_ctx, case_id='demo_prohibited')
print(f'4. Prohibited phrase   → overall_pass: {r4.overall_pass}')
print(f'   safety_pass: {r4.safety_pass}')
print(f'   failure_reasons: {[x for x in r4.failure_reasons if "prohibited" in x]}')

In [ ]:
# 5. Privacy suppression not disclosed
if suppressed:
    supp_ctx = suppressed[0]
    no_disclosure = dict(valid_insight,
        engagement_insight='10 active users with 70% returning share.',
        evidence_limitations=['No significant limitations.']
    )
    r5 = evaluate_report_insight(no_disclosure, supp_ctx, case_id='demo_privacy')
    print(f'5. Privacy not disclosed → overall_pass: {r5.overall_pass}')
    print(f'   evidence_disclosure_pass: {r5.evidence_disclosure_pass}')
    print(f'   failure_reasons: {[x for x in r5.failure_reasons if "privac" in x.lower()]}')
else:
    print('5. No suppressed contexts in this mart — skip.')

In [ ]:
# 6. Deterministic fallback (all current stored outputs)
from src.genai.insight_generator import generate_rule_based_insight
fallback = generate_rule_based_insight(demo_ctx)
print('6. Deterministic fallback for', demo_ctx['report_id'])
for f in ['executive_summary', 'recommended_action', 'generation_status']:
    print(f'   [{f}] {fallback.get(f)}')

## 9. Evaluation Framework

All checks are deterministic — no LLM-as-judge.

| Dimension | Type | Hard-failure condition |
|-----------|------|------------------------|
| Completeness | Hard | Required field missing or empty |
| Safety | Hard | Prohibited phrase detected |
| Direction | Hard | Language contradicts context status |
| Numerical | Hard | % or count claim > ±5 from any context value |
| Action alignment | Hard | Generated action does not preserve deterministic action |
| Evidence disclosure | Hard | Required limitation not disclosed |
| Readability | Soft | Heuristic word-count check; threshold ≥ 0.5 |
| Conciseness | Soft | Informational only |

**Overall pass** = all 6 hard dimensions True AND readability ≥ 0.5

In [ ]:
from src.genai.evaluation import run_regression_against_stored_outputs

eval_summary = run_regression_against_stored_outputs(project_root=PROJECT_ROOT)
print(f"Cases evaluated:    {eval_summary['cases_evaluated']}")
print(f"Overall pass rate:  {eval_summary['overall_pass_rate']}")
print(f"By status:          {eval_summary['by_generation_status']}")
print()
print(eval_summary['evaluation_note'])

In [ ]:
from src.genai.evaluation import load_evaluation_cases, load_golden_outputs

cases = load_evaluation_cases()
goldens = load_golden_outputs()
report_cases    = [c for c in cases if c['insight_type'] == 'report']
portfolio_cases = [c for c in cases if c['insight_type'] == 'portfolio']
print(f'Fixture cases:   {len(cases)} total ({len(report_cases)} report, {len(portfolio_cases)} portfolio)')
print(f'Golden configs:  {len(goldens)}')

pass_expected = [c for c in cases if c.get('expected_validation_outcome') == 'pass']
fail_expected = [c for c in cases if c.get('expected_validation_outcome') == 'fail']
print(f'Expected pass:   {len(pass_expected)}')
print(f'Expected fail:   {len(fail_expected)}  (safety, numerical, direction)')

## 10. Cost Control and Hash Reuse

Before each API call the pipeline computes `SHA-256(sorted context + prompt_version + model_name)`. If the hash matches a prior `success` or `reused` result, the output is returned unchanged with no API call.

In [ ]:
from src.genai.insight_generator import _compute_context_hash, DEFAULT_MODEL
from src.genai.portfolio_insights import _compute_portfolio_hash
from src.genai.prompts import REPORT_INSIGHT_PROMPT_VERSION

ctx = contexts[0]
h_base    = _compute_context_hash(ctx, REPORT_INSIGHT_PROMPT_VERSION, DEFAULT_MODEL)
h_ctx     = _compute_context_hash({**ctx, 'recent_28d_views': 999}, REPORT_INSIGHT_PROMPT_VERSION, DEFAULT_MODEL)
h_prompt  = _compute_context_hash(ctx, 'report_insight_v2', DEFAULT_MODEL)
h_model   = _compute_context_hash(ctx, REPORT_INSIGHT_PROMPT_VERSION, 'gpt-4o')

print(f'Stable (same inputs):              {h_base[:20]}...')
print(f'Changed context   → new hash:      {h_ctx[:20]}...')
print(f'Changed prompt    → new hash:      {h_prompt[:20]}...')
print(f'Changed model     → new hash:      {h_model[:20]}...')
print()

# Portfolio: shortlist is excluded from hash
pctx_a = {**portfolio_ctx, 'attention_shortlist': []}
pctx_b = {**portfolio_ctx, 'attention_shortlist': [{'report_id': 'X'}]}
ph_a = _compute_portfolio_hash(pctx_a, PORTFOLIO_INSIGHT_PROMPT_VERSION, DEFAULT_MODEL)
ph_b = _compute_portfolio_hash(pctx_b, PORTFOLIO_INSIGHT_PROMPT_VERSION, DEFAULT_MODEL)
print(f'Portfolio: shortlist change → same hash: {ph_a == ph_b}  (shortlist excluded from hash)')
print()
print('Reuse applies only to prior outputs with generation_status in {success, reused}.')
print('Rule-based outputs are regenerated on each run (no API cost).')

from src.genai.prompts import PORTFOLIO_INSIGHT_PROMPT_VERSION

## 11. Current Limitations

1. **Synthetic source data.** All mart inputs are from a synthetic dataset. Insights describe synthetic patterns, not real reports or users.

2. **Insufficient model-health evidence.** All 30 reports have `model_diagnostic_status = insufficient_evidence`. Model-health commentary is limited to acknowledging this gap. This is expected — no production forecast backtests have been run.

3. **Rule-based generation only in this environment.** No API key is configured. All stored outputs have `generation_status = rule_based`. The live LLM path has not been exercised and has not been validated on these outputs.

4. **Shortlist cap.** The attention shortlist shows at most 5 reports. In a larger portfolio with many high-priority reports, the shortlist is not the complete action queue.

5. **Portfolio insights not shown in Streamlit.** `src/app/streamlit_app.py` displays report-level insights only. Portfolio insights are persisted to JSON but not yet rendered. This is deferred to Sprint 9.

6. **Stakeholder usefulness testing not completed.** The human-review rubric (`docs/genai_evaluation_rubric.md`) is defined but no formal usefulness experiment has been conducted.

7. **Operational observability is partial.** API latency, token usage, and cost are not yet instrumented. See `docs/genai_operations.md`.

## 12. Output Files

In [ ]:
canonical_outputs = [
    PROJECT_ROOT / 'outputs' / 'insights'   / 'report_ai_insights.json',
    PROJECT_ROOT / 'outputs' / 'insights'   / 'portfolio_ai_insight.json',
    PROJECT_ROOT / 'outputs' / 'evaluation' / 'genai_evaluation_results.csv',
    PROJECT_ROOT / 'outputs' / 'evaluation' / 'genai_evaluation_summary.json',
]
for p in canonical_outputs:
    exists = p.exists()
    size   = f'{p.stat().st_size:,} bytes' if exists else 'NOT FOUND'
    status = '✓' if exists else '✗'
    print(f'{status}  {p.relative_to(PROJECT_ROOT)}  ({size})')